In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
import math, itertools, warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
url = 'https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv'
df = pd.read_csv(url)
print(df.shape) # expected (1338, 7)
print(df.dtypes)
print(df.head())

## Part 1 - Fitting Distributions to the Data
### 1.1 The Normal Distribution - BMI

In [ ]:
# Task 1: Compute sample mean and std of BMI to 4 decimal places
mu = df['bmi'].mean()
sigma = df['bmi'].std()
print(f"Sample mean (μ): {mu:.4f}")
print(f"Sample standard deviation (σ): {sigma:.4f}")

In [ ]:
# Task 2: Compute predicted probabilities analytically
p_under_normal = stats.norm.cdf(25, loc=mu, scale=sigma)
p_overweight = stats.norm.cdf(30, loc=mu, scale=sigma) - stats.norm.cdf(25, loc=mu, scale=sigma)
p_obese = 1 - stats.norm.cdf(30, loc=mu, scale=sigma)

print(f"P(BMI < 25): {p_under_normal:.4f}")
print(f"P(25 <= BMI < 30): {p_overweight:.4f}")
print(f"P(BMI >= 30): {p_obese:.4f}")

In [ ]:
# Task 3: Compute actual proportions and display side by side
n_total = len(df)
actual_under_normal = len(df[df['bmi'] < 25]) / n_total
actual_overweight = len(df[(df['bmi'] >= 25) & (df['bmi'] < 30)]) / n_total
actual_obese = len(df[df['bmi'] >= 30]) / n_total

comparison_df = pd.DataFrame({
    'Range': ['BMI < 25 (Under/Normal)', '25 <= BMI < 30 (Overweight)', 'BMI >= 30 (Obese)'],
    'Predicted Proportion': [p_under_normal, p_overweight, p_obese],
    'Actual Proportion': [actual_under_normal, actual_overweight, actual_obese]
})
display(comparison_df)

**Interpretation (Task 3):** The predictions are very close to the actual proportions. The Normal model slightly underestimates the extreme tails, but generally serves as a solid fit for BMI in this dataset.

In [ ]:
# Task 4: Plot histogram of BMI with fitted Normal PDF overlaid
plt.figure(figsize=(10, 6))
sns.histplot(df['bmi'], stat='density', bins=30, alpha=0.5, label='Actual BMI Data')
xmin, xmax = plt.xlim()
x = np.linspace(xmin, xmax, 100)
p = stats.norm.pdf(x, mu, sigma)
plt.plot(x, p, 'k', linewidth=2, label='Fitted Normal PDF')
plt.title('Histogram of BMI with Fitted Normal Distribution')
plt.xlabel('BMI')
plt.ylabel('Density')
plt.legend()
plt.show()

**Interpretation (Task 4):** Visually, the Normal distribution fits the central mass of the data quite well, particularly around the mean (BMI ~ 30). However, it does not fit perfectly in the right tail; there is a slight positive skew in the actual data with more extreme high-BMI outliers than the perfect Normal model predicts.

In [ ]:
# Task 5: Compute 5th and 95th percentile
p5_norm = stats.norm.ppf(0.05, loc=mu, scale=sigma)
p95_norm = stats.norm.ppf(0.95, loc=mu, scale=sigma)

p5_actual = np.percentile(df['bmi'], 5)
p95_actual = np.percentile(df['bmi'], 95)

print(f"5th Percentile - Normal: {p5_norm:.4f}, Actual: {p5_actual:.4f}")
print(f"95th Percentile - Normal: {p95_norm:.4f}, Actual: {p95_actual:.4f}")
print(f"Difference at 95th Percentile: {abs(p95_norm - p95_actual):.4f}")

**Interpretation (Task 5):** The Normal model underestimates the 95th percentile by roughly 0.43 units. While it is a relatively small absolute difference, it confirms the right skew seen in the histogram—the real data has slightly more extreme obesity cases than a theoretical Normal distribution expects.

### 1.2 The Binomial Distribution - Smoker Rate

In [ ]:
# Task 6: Compute empirical probability p
p_smoker = len(df[df['smoker'] == 'yes']) / len(df)
print(f"Empirical probability of being a smoker (p): {p_smoker:.4f}")

In [ ]:
# Task 7: Binomial PMF manual vs scipy
n = 50
p = p_smoker

results = []
for k in [5, 10, 15, 20]:
    manual = math.comb(n, k) * (p**k) * ((1-p)**(n-k))
    scipy_pmf = stats.binom.pmf(k, n, p)
    results.append({'k (smokers)': k, 'Manual PMF': manual, 'SciPy PMF': scipy_pmf})

pmf_df = pd.DataFrame(results)
display(pmf_df)

In [ ]:
# Task 8: Binomial CDF
p_fewer_than_8 = stats.binom.cdf(7, n, p)
p_more_than_20 = 1 - stats.binom.cdf(20, n, p)

print(f"P(fewer than 8 smokers in 50): {p_fewer_than_8 * 100:.2f}%")
print(f"P(more than 20 smokers in 50): {p_more_than_20 * 100:.2f}%")

**Interpretation (Task 8):** For a pricing manager reviewing groups of 50 policyholders, there is roughly an 18.5% chance that a group will have exceptionally few smokers (fewer than 8), representing a highly profitable, low-risk group. Conversely, there is only a 0.16% chance of having more than 20 smokers, meaning it is exceedingly rare to encounter a group of 50 with an overwhelming majority of high-cost smokers.

In [ ]:
# Task 9: Simulate 10,000 groups
simulations = np.random.binomial(n=50, p=p_smoker, size=10000)
sim_mean = np.mean(simulations)
sim_var = np.var(simulations)

theoretical_mean = n * p_smoker
theoretical_var = n * p_smoker * (1 - p_smoker)

print(f"Simulated Mean: {sim_mean:.4f} | Theoretical Mean: {theoretical_mean:.4f}")
print(f"Simulated Variance: {sim_var:.4f} | Theoretical Variance: {theoretical_var:.4f}")
print(f"Mean % Error: {abs(sim_mean - theoretical_mean) / theoretical_mean * 100:.2f}%")
print(f"Variance % Error: {abs(sim_var - theoretical_var) / theoretical_var * 100:.2f}%")

In [ ]:
# Task 10: Smoker rates by sex
p_male_smoker = len(df[(df['sex'] == 'male') & (df['smoker'] == 'yes')]) / len(df[df['sex'] == 'male'])
p_female_smoker = len(df[(df['sex'] == 'female') & (df['smoker'] == 'yes')]) / len(df[df['sex'] == 'female'])

sex_results = []
for sex, p_sex in [('Male', p_male_smoker), ('Female', p_female_smoker)]:
    expected = n * p_sex
    p_zero = stats.binom.pmf(0, n, p_sex)
    sex_results.append({'Sex': sex, 'Expected Smokers (n=50)': expected, 'P(0 smokers)': p_zero})

display(pd.DataFrame(sex_results))

### 1.3 The Poisson Distribution - Number of Dependants

In [ ]:
# Task 11: Poisson mean, variance, dispersion
lam = df['children'].mean()
var_children = df['children'].var(ddof=0)
dispersion_ratio = var_children / lam

print(f"Mean (λ): {lam:.4f}")
print(f"Variance: {var_children:.4f}")
print(f"Dispersion Ratio: {dispersion_ratio:.4f}")

**Interpretation (Task 11):** The dispersion ratio is ~1.32. Because the variance exceeds the mean (ratio > 1), the data is slightly overdispersed. While Poisson is a reasonable starting point for count data, the overdispersion suggests the variance is wider than a strict Poisson model assumes, likely due to a high number of zero-children policies.

In [ ]:
# Task 12: Poisson PMF manual vs scipy
poisson_results = []
for k in range(5):
    manual = (lam**k * math.exp(-lam)) / math.factorial(k)
    scipy_pmf = stats.poisson.pmf(k, mu=lam)
    poisson_results.append({'k (children)': k, 'Manual PMF': manual, 'SciPy PMF': scipy_pmf})

display(pd.DataFrame(poisson_results).round(4))

In [ ]:
# Task 13: Compare Poisson to actual
actual_props = []
for k in range(5):
    prop = len(df[df['children'] == k]) / len(df)
    actual_props.append(prop)

poisson_df = pd.DataFrame(poisson_results)
poisson_df['Actual Proportion'] = actual_props
poisson_df['Absolute Gap'] = abs(poisson_df['Actual Proportion'] - poisson_df['SciPy PMF'])
display(poisson_df[['k (children)', 'Actual Proportion', 'SciPy PMF', 'Absolute Gap']])

**Interpretation (Task 13):** The value `k=0` has the largest gap between the model and reality. The actual proportion of policies with 0 children is much higher than the Poisson distribution predicts, confirming zero-inflation.

In [ ]:
# Task 14: P(children >= 3)
p_ge_3_model = 1 - stats.poisson.cdf(2, mu=lam)
actual_ge_3 = len(df[df['children'] >= 3]) / len(df)
error = abs(p_ge_3_model - actual_ge_3) / actual_ge_3 * 100

print(f"P(children >= 3) - Poisson Model: {p_ge_3_model:.4f}")
print(f"P(children >= 3) - Actual Data: {actual_ge_3:.4f}")
print(f"Affected Policyholders: {len(df[df['children'] >= 3])}")
print(f"Percentage Error: {error:.2f}%")

## Part 2 - Hypothesis Testing

### 2.1 Do Smokers Cost the Company Significantly More?

**Expectation of Outcome:** Given the well-known health risks associated with smoking, I strongly expect that smokers generate substantially higher medical charges compared to non-smokers. This hypothesis test should result in a highly significant p-value, confirming the need for a separate risk pricing tier.

In [ ]:
# Task 15: Descriptive stats by smoker status
smokers = df[df['smoker'] == 'yes']['charges']
nonsmokers = df[df['smoker'] == 'no']['charges']

stats_df = pd.DataFrame({
    'Smokers': [len(smokers), smokers.mean(), smokers.median(), smokers.std()],
    'Non-Smokers': [len(nonsmokers), nonsmokers.mean(), nonsmokers.median(), nonsmokers.std()]
}, index=['n', 'mean', 'median', 'std'])
display(stats_df)

In [ ]:
# Task 16: Box plot
plt.figure(figsize=(8, 6))
sns.boxplot(x='smoker', y='charges', data=df)
plt.title('Distribution of Charges by Smoker Status')
plt.xlabel('Smoker Status')
plt.ylabel('Annual Charges')
plt.show()

**Interpretation (Task 16):** The box plot reveals that not only are the median charges drastically higher for smokers, but the spread (variance) is also considerably larger. Non-smokers have a tight cluster of low charges with some outliers, whereas smokers are distributed across a much wider and higher range. This unequal variance strongly suggests we must use Welch's t-test.

In [ ]:
# Task 17: Levene's Test
stat, p_levene = stats.levene(smokers, nonsmokers)
print(f"Levene's test statistic: {stat:.4f}, p-value: {p_levene:.4e}")

**Interpretation (Task 17):** The p-value for Levene's test is virtually 0, which is far below 0.05. This means we reject the null hypothesis of equal variances. Consequently, we will use Welch's t-test (`equal_var=False`) which is robust against unequal variances.

In [ ]:
# Task 18: Welch's t-test
t_stat, p_ttest = stats.ttest_ind(smokers, nonsmokers, equal_var=False)
print(f"Welch's t-statistic: {t_stat:.4f}, p-value: {p_ttest:.4e}")

**Interpretation (Task 18):** At α = 0.05, we reject the null hypothesis since p < 0.05. This means mean charges differ significantly between smokers and non-smokers. For the pricing team, this justifies charging a substantially higher premium for smokers due to their statistically proven higher cost burden.

In [ ]:
# Task 19: 95% CI
mean_diff = smokers.mean() - nonsmokers.mean()
n1, n2 = len(smokers), len(nonsmokers)
s1, s2 = smokers.std(), nonsmokers.std()
se = np.sqrt(s1**2/n1 + s2**2/n2)
df_welch = ((s1**2/n1 + s2**2/n2)**2) / ((s1**2/n1)**2/(n1-1) + (s2**2/n2)**2/(n2-1))

ci_lower, ci_upper = stats.t.interval(0.95, df=df_welch, loc=mean_diff, scale=se)
print(f"95% CI for difference: ({ci_lower:.2f}, {ci_upper:.2f})")

**Interpretation (Task 19):** We can be 95% confident that the company can expect smokers to cost between $22,197.21 and $25,034.71 more per year than non-smokers.

In [ ]:
# Task 20: Cohen's d
sp = np.sqrt(((n1-1)*s1**2 + (n2-1)*s2**2) / (n1+n2-2))
d = mean_diff / sp
print(f"Cohen's d: {d:.4f}")

**Interpretation (Task 20):** A Cohen's d of 3.16 is considered an extremely large effect size (d > 0.8). This indicates that the difference is not only statistically significant but also highly practically significant, definitively justifying a different pricing tier.

In [ ]:
# Task 21: Mann-Whitney U Test
u_stat, p_mwu = stats.mannwhitneyu(smokers, nonsmokers, alternative='two-sided')
print(f"Mann-Whitney U: {u_stat}, p-value: {p_mwu:.4e}")

**Interpretation (Task 21):** The Mann-Whitney U test also yields a highly significant p-value (p < 0.05), perfectly agreeing with the t-test conclusion despite the heavy right-skew of the charges column.

### 2.2 Is There a Statistically Significant Difference Between Male and Female Charges?

In [ ]:
# Task 22: Charges for males and females
males = df[df['sex'] == 'male']['charges']
females = df[df['sex'] == 'female']['charges']

sex_stats = pd.DataFrame({
    'Male': [len(males), males.mean(), males.std()],
    'Female': [len(females), females.mean(), females.std()]
}, index=['n', 'mean', 'std'])
display(sex_stats)

**Interpretation (Task 22):** The descriptive data suggests a slight difference, with males costing roughly $1,387 more on average than females, though both groups have high standard deviations indicating large overlap.

In [ ]:
# Task 23: Levene's and t-test
stat_sex, p_levene_sex = stats.levene(males, females)
print(f"Levene's p-value: {p_levene_sex:.4f}")
# p < 0.05, so use Welch's
t_sex, p_t_sex = stats.ttest_ind(males, females, equal_var=False)
print(f"Welch's t-statistic: {t_sex:.4f}, p-value: {p_t_sex:.4f}")

**Interpretation (Task 24):** At α = 0.05, we reject the null hypothesis (p = 0.0358 < 0.05). There is a statistically significant difference in mean charges between males and females.

In [ ]:
# Task 24 CI: 95% CI
mean_diff_sex = males.mean() - females.mean()
n_m, n_f = len(males), len(females)
s_m, s_f = males.std(), females.std()
se_sex = np.sqrt(s_m**2/n_m + s_f**2/n_f)
df_sex = ((s_m**2/n_m + s_f**2/n_f)**2) / ((s_m**2/n_m)**2/(n_m-1) + (s_f**2/n_f)**2/(n_f-1))

ci_low_sex, ci_high_sex = stats.t.interval(0.95, df=df_sex, loc=mean_diff_sex, scale=se_sex)
print(f"95% CI for difference (Male - Female): ({ci_low_sex:.2f}, {ci_high_sex:.2f})")

**Interpretation (Task 24 continued):** The confidence interval is ($91.86, $2682.49). Because the interval does not include zero, it corroborates the significant t-test result, proving that males cost strictly more on average than females.

In [ ]:
# Task 25: Cohen's d
sp_sex = np.sqrt(((n_m-1)*s_m**2 + (n_f-1)*s_f**2) / (n_m+n_f-2))
d_sex = mean_diff_sex / sp_sex
print(f"Cohen's d (Male vs Female): {d_sex:.4f}")

**Interpretation (Task 25):** The effect size is only 0.1147, which is classified as "negligible" (|d| < 0.2). Despite the statistical significance likely driven by the large sample size, the practical difference is trivial. Therefore, I would **not** recommend treating sex as a primary pricing factor, as the actual difference in risk and cost is negligible.

### 2.3 Has BMI Increased With Age?

In [ ]:
# Task 26: BMI by Age group
under_40 = df[df['age'] < 40]['bmi']
over_40 = df[df['age'] >= 40]['bmi']

age_stats = pd.DataFrame({
    'Under 40': [len(under_40), under_40.mean(), under_40.std()],
    'Over 40': [len(over_40), over_40.mean(), over_40.std()]
}, index=['n', 'mean', 'std'])
display(age_stats)

In [ ]:
# Task 27: Levene's and t-test
stat_age, p_levene_age = stats.levene(over_40, under_40)
print(f"Levene's p-value: {p_levene_age:.4f}")
# p >= 0.05, so use standard t-test
t_age, p_t_age = stats.ttest_ind(over_40, under_40, equal_var=True)
print(f"Standard t-statistic: {t_age:.4f}, p-value: {p_t_age:.4f}")

In [ ]:
# Task 28: 95% CI and Cohen's d
mean_diff_age = over_40.mean() - under_40.mean()
n_o, n_u = len(over_40), len(under_40)
s_o, s_u = over_40.std(), under_40.std()

se_age = np.sqrt(s_o**2/n_o + s_u**2/n_u)
ci_low_age, ci_high_age = stats.t.interval(0.95, df=n_o+n_u-2, loc=mean_diff_age, scale=se_age)
print(f"95% CI for difference (Over 40 - Under 40): ({ci_low_age:.2f}, {ci_high_age:.2f})")

sp_age = np.sqrt(((n_o-1)*s_o**2 + (n_u-1)*s_u**2) / (n_o+n_u-2))
d_age = mean_diff_age / sp_age
print(f"Cohen's d: {d_age:.4f}")

**Interpretation (Task 28):** At α = 0.05, the p-value (0.0004) is statistically significant. However, the Cohen's d of 0.195 indicates a negligible effect size (|d| < 0.2). Thus, while older policyholders technically have a higher BMI, the difference is practically negligible.

In [ ]:
# Task 29: Pearson's correlation
r, p_corr = stats.pearsonr(df['age'], df['bmi'])
print(f"Pearson's r: {r:.4f}, p-value: {p_corr:.4e}")

**Interpretation (Task 29):** The positive correlation coefficient (r = 0.1093) with a significant p-value perfectly agrees with the t-test conclusion: there is a significant but extremely weak relationship between age and BMI. Both approaches give consistent results because they are fundamentally detecting the same slight upward trend in BMI as age increases.

### 2.4 Regional Pricing - Multiple Comparisons

In [ ]:
# Task 30: Mean charges by region
region_stats = df.groupby('region')['charges'].agg(['mean', 'std']).sort_values(by='mean', ascending=False)
display(region_stats)

In [ ]:
# Task 31 & 32: Pairwise t-tests and Bonferroni correction
regions = df['region'].unique()
results = []

for r1, r2 in itertools.combinations(regions, 2):
    g1 = df[df['region']==r1]['charges']
    g2 = df[df['region']==r2]['charges']
    t_val, p_val = stats.ttest_ind(g1, g2, equal_var=False)
    results.append({
        'pair': f'{r1} vs {r2}', 
        't_stat': t_val, 
        'p_value': p_val,
        'adjusted_p': min(p_val * 6, 1.0)
    })

pairwise_df = pd.DataFrame(results).sort_values('p_value', ascending=True)
display(pairwise_df)

sig_before = len(pairwise_df[pairwise_df['p_value'] < 0.05])
sig_after = len(pairwise_df[pairwise_df['adjusted_p'] < 0.05])
print(f"Significant pairs before correction: {sig_before}")
print(f"Significant pairs after correction: {sig_after}")

In [ ]:
# Task 33: Family-wise error rate
fwer = 1 - (1 - 0.05)**6
print(f"FWER = {fwer:.4f}")

**Interpretation (Task 33):** The FWER of ~0.265 means there is a 26.5% chance of making at least one Type I error (false positive) when running 6 independent t-tests without correction. The more tests you run, the higher the probability that random sampling noise will falsely appear as a significant difference, which is why the Bonferroni adjustment is required.

In [ ]:
# Task 34: 95% CI for each region
ci_results = []
for region in regions:
    data = df[df['region']==region]['charges']
    n_reg = len(data)
    mean_reg = data.mean()
    se_reg = data.std() / np.sqrt(n_reg)
    ci_low, ci_high = stats.t.interval(0.95, df=n_reg-1, loc=mean_reg, scale=se_reg)
    ci_results.append({'region': region, 'mean': mean_reg, 'CI_lower': ci_low, 'CI_upper': ci_high})

ci_df = pd.DataFrame(ci_results)
display(ci_df)

**Interpretation (Task 34):** The 'southeast' region has the widest Confidence Interval. CI width is driven by the standard deviation and the sample size. Although sample sizes are similar across regions, the 'southeast' region has the highest standard deviation in charges, resulting in the widest margin of error.

In [ ]:
# Task 35: Error bar chart
plt.figure(figsize=(10, 6))
grand_mean = df['charges'].mean()

x_means = ci_df['mean']
y_pos = np.arange(len(ci_df['region']))
x_err = [ci_df['mean'] - ci_df['CI_lower'], ci_df['CI_upper'] - ci_df['mean']]

plt.errorbar(x_means, y_pos, xerr=x_err, fmt='o', capsize=5)
plt.yticks(y_pos, ci_df['region'])
plt.axvline(x=grand_mean, color='red', linestyle='--', label='Grand Mean')

plt.title('95% Confidence Intervals for Regional Mean Charges')
plt.xlabel('Mean Charges')
plt.ylabel('Region')
plt.legend()
plt.show()

**Interpretation (Task 35):** Looking at the horizontal error bars, almost all the regional confidence intervals overlap substantially. The only pair that borders on non-overlap is Southeast vs. Southwest. This visual overlap precisely matches our Bonferroni-corrected t-test results: there are no statistically significant differences in mean charges across any region pairs after correction.

## Part 3 - Your Own Investigation

### 3.1 Open-Ended Question from the Manager

In [ ]:
# Tasks 36-39: Sex-Stratified Smoking Effect
males_df = df[df['sex']=='male']
females_df = df[df['sex']=='female']

res_list = []

# Male pipeline
m_smokers = males_df[males_df['smoker']=='yes']['charges']
m_nonsmokers = males_df[males_df['smoker']=='no']['charges']
_, p_lev_m = stats.levene(m_smokers, m_nonsmokers) # p < 0.05
t_m, p_m = stats.ttest_ind(m_smokers, m_nonsmokers, equal_var=False)

diff_m = m_smokers.mean() - m_nonsmokers.mean()
n1_m, n2_m = len(m_smokers), len(m_nonsmokers)
s1_m, s2_m = m_smokers.std(), m_nonsmokers.std()
se_m = np.sqrt(s1_m**2/n1_m + s2_m**2/n2_m)
df_w_m = ((s1_m**2/n1_m + s2_m**2/n2_m)**2) / ((s1_m**2/n1_m)**2/(n1_m-1) + (s2_m**2/n2_m)**2/(n2_m-1))
ci_m_l, ci_m_u = stats.t.interval(0.95, df=df_w_m, loc=diff_m, scale=se_m)

sp_m = np.sqrt(((n1_m-1)*s1_m**2 + (n2_m-1)*s2_m**2) / (n1_m+n2_m-2))
d_m = diff_m / sp_m

# Female pipeline
f_smokers = females_df[females_df['smoker']=='yes']['charges']
f_nonsmokers = females_df[females_df['smoker']=='no']['charges']
_, p_lev_f = stats.levene(f_smokers, f_nonsmokers) # p < 0.05
t_f, p_f = stats.ttest_ind(f_smokers, f_nonsmokers, equal_var=False)

diff_f = f_smokers.mean() - f_nonsmokers.mean()
n1_f, n2_f = len(f_smokers), len(f_nonsmokers)
s1_f, s2_f = f_smokers.std(), f_nonsmokers.std()
se_f = np.sqrt(s1_f**2/n1_f + s2_f**2/n2_f)
df_w_f = ((s1_f**2/n1_f + s2_f**2/n2_f)**2) / ((s1_f**2/n1_f)**2/(n1_f-1) + (s2_f**2/n2_f)**2/(n2_f-1))
ci_f_l, ci_f_u = stats.t.interval(0.95, df=df_w_f, loc=diff_f, scale=se_f)

sp_f = np.sqrt(((n1_f-1)*s1_f**2 + (n2_f-1)*s2_f**2) / (n1_f+n2_f-2))
d_f = diff_f / sp_f

res_list.append({'Sex': 'Male', 'Mean Difference': diff_m, 'CI Lower': ci_m_l, 'CI Upper': ci_m_u, "Cohen's d": d_m, 'Raw p-value': p_m, 'Bonferroni p-value': min(p_m*2, 1.0)})
res_list.append({'Sex': 'Female', 'Mean Difference': diff_f, 'CI Lower': ci_f_l, 'CI Upper': ci_f_u, "Cohen's d": d_f, 'Raw p-value': p_f, 'Bonferroni p-value': min(p_f*2, 1.0)})

stratified_df = pd.DataFrame(res_list)
display(stratified_df)

**Memo to Manager (Task 40):** 
Yes, the smoking premium is remarkably consistent and massive across both sexes. For male policyholders, smoking adds an average of $24,955 to their costs with a 95% confidence interval of ($23,129 to $26,781). For female policyholders, smoking adds $21,917 to their costs with an interval of ($19,660 to $24,173). In both cases, the Bonferroni-corrected p-value is extremely significant (near 0) and the effect sizes are colossal (Cohen's d ≈ 3.3 for males, 2.9 for females). Ultimately, while men show a slightly larger absolute smoking premium, the practical takeaway for pricing is identical: smoking is the dominant risk factor for medical costs regardless of sex, and both sexes require the highest pricing tiers if they smoke.

### 3.2 Your Own Question

**Question & Hypothesis (Task 41):** 
*Question:* Do policyholders in the highest BMI quartile have significantly higher charges than the rest, even after excluding smokers?
*Null Hypothesis (H₀):* There is no significant difference in mean charges between non-smokers in the top BMI quartile and non-smokers in the lower three BMI quartiles.
*Alternative Hypothesis (H₁):* Non-smokers in the top BMI quartile have significantly higher mean charges than non-smokers in the lower three BMI quartiles.

In [ ]:
# Filter to only non-smokers to remove the smoking confounding variable
nonsmokers_df = df[df['smoker'] == 'no']

# Determine the 75th percentile for BMI among non-smokers
bmi_75th = np.percentile(nonsmokers_df['bmi'], 75)

# Split into two groups
high_bmi = nonsmokers_df[nonsmokers_df['bmi'] >= bmi_75th]['charges']
low_bmi = nonsmokers_df[nonsmokers_df['bmi'] < bmi_75th]['charges']

# Task 42: Descriptive Stats
custom_stats = pd.DataFrame({
    'Top BMI Quartile': [len(high_bmi), high_bmi.mean(), high_bmi.std()],
    'Lower 3 BMI Quartiles': [len(low_bmi), low_bmi.mean(), low_bmi.std()]
}, index=['n', 'mean', 'std'])
display(custom_stats)

# Task 43: Assumption Check & Test
stat_custom, p_lev_custom = stats.levene(high_bmi, low_bmi)
print(f"Levene's p-value: {p_lev_custom:.4f}")
# p-value < 0.05, so use Welch's t-test
t_custom, p_t_custom = stats.ttest_ind(high_bmi, low_bmi, equal_var=False)
print(f"Welch's t-statistic: {t_custom:.4f}, p-value: {p_t_custom:.4e}")

# Task 44: 95% CI
diff_custom = high_bmi.mean() - low_bmi.mean()
n_h, n_l = len(high_bmi), len(low_bmi)
s_h, s_l = high_bmi.std(), low_bmi.std()
se_custom = np.sqrt(s_h**2/n_h + s_l**2/n_l)
df_w_custom = ((s_h**2/n_h + s_l**2/n_l)**2) / ((s_h**2/n_h)**2/(n_h-1) + (s_l**2/n_l)**2/(n_l-1))

ci_l_custom, ci_u_custom = stats.t.interval(0.95, df=df_w_custom, loc=diff_custom, scale=se_custom)
print(f"95% CI for difference: ({ci_l_custom:.2f}, {ci_u_custom:.2f})")

# Task 45: Effect Size
sp_custom = np.sqrt(((n_h-1)*s_h**2 + (n_l-1)*s_l**2) / (n_h+n_l-2))
d_custom = diff_custom / sp_custom
print(f"Cohen's d: {d_custom:.4f}")

**Conclusion (Task 46):** 
I found that even when we entirely exclude the massive effect of smoking, extreme obesity (being in the top 25% of BMI) still drives significantly higher medical charges. The Welch's t-test yielded a highly significant p-value (p < 0.05), rejecting the null hypothesis. The 95% confidence interval shows that top-quartile BMI non-smokers cost between $1,215 and $2,787 more annually than their lower BMI counterparts, with a moderate effect size (Cohen's d = 0.38). The company can use this information to create a secondary pricing adjustment for extreme BMI levels, capturing additional risk premiums from policyholders who do not smoke but still pose higher costs due to weight-related health risks.